# 4x4 Toric Code: SR and NES Variance Tests

This notebook runs the same 4x4 toric-code experiment for `k=2`, `k=4`, and `k=5`:

1. SR only, constant learning rate.
2. SR plus a cosine learning-rate routine.

The training logs include the old Ritz-span energies plus the NES S12-style variance estimates from the local energy matrix.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

# Change this if needed.
PROJECT_ROOT = Path.home() / "Desktop" / "Master Thesis" / "NES_Spins"
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from nes_lattice.train import TrainConfig, train, save_history, load_history

In [ ]:
def toric_sr_cfg(k, *, schedule="constant", seed=0, steps=2500, tag=""):
    return TrainConfig(
        shape=(4, 4),
        hamiltonian="toric_code",
        k=k,
        Je=1.0,
        Jm=1.0,

        # Start with the current raw-spin ViT.  You can swap to rbm/cnn/toric_rbm
        # if your local models.py supports them.
        model="vit",
        vit_patch_size=1,
        vit_d_model=32,
        vit_num_layers=2,
        vit_num_heads=4,
        vit_mlp_ratio=2,
        vit_use_positional_embeddings=True,
        vit_log_amplitude_clip=20.0,
        init_scale=0.02,

        optimizer="sr",
        sr_diag_shift=1e-3,
        sr_cg_iters=40,
        sr_cg_tol=1e-5,

        steps=steps,
        lr=2e-3,
        lr_schedule=schedule,
        lr_final_factor=0.05,
        grad_clip=None,

        n_chains=96,
        n_samples=8,
        sweep_steps=32,
        burn_in=128,

        # For a raw-spin ansatz loop moves help sector mixing.  Single flips are
        # off here to focus first on ground-sector dynamics; for k=5 you can later
        # try a small single_flip probability if the fifth state is bad.
        toric_loop_prob=0.10,
        toric_single_flip_prob=0.0,
        toric_cover_sectors=True,

        print_every=250,
        eval_exact_if_sites_leq=12,
        eval_chains=256,
        eval_samples=64,
        reference="auto",
        own_ed_max_sites=14,
        jitter=1e-6,

        log_variance=True,
        variance_every=250,

        seed=seed,
    )


def run_and_save(cfg, name):
    params, history = train(cfg)
    path = PROJECT_ROOT / "results" / f"{name}.json"
    save_history(history, path, cfg)
    print("saved", path)
    return params, history, path

## k=2: SR only

In [ ]:
cfg_k2_sr = toric_sr_cfg(2, schedule="constant", seed=2, steps=2500)
params_k2_sr, hist_k2_sr, path_k2_sr = run_and_save(cfg_k2_sr, "toric4x4_k2_vit_sr")

## k=2: SR + cosine LR routine

In [ ]:
cfg_k2_sr_sched = toric_sr_cfg(2, schedule="cosine", seed=2, steps=2500)
params_k2_sr_sched, hist_k2_sr_sched, path_k2_sr_sched = run_and_save(cfg_k2_sr_sched, "toric4x4_k2_vit_sr_cosine")

## k=4: SR only

In [ ]:
cfg_k4_sr = toric_sr_cfg(4, schedule="constant", seed=4, steps=3000)
params_k4_sr, hist_k4_sr, path_k4_sr = run_and_save(cfg_k4_sr, "toric4x4_k4_vit_sr")

## k=4: SR + cosine LR routine

In [ ]:
cfg_k4_sr_sched = toric_sr_cfg(4, schedule="cosine", seed=4, steps=3000)
params_k4_sr_sched, hist_k4_sr_sched, path_k4_sr_sched = run_and_save(cfg_k4_sr_sched, "toric4x4_k4_vit_sr_cosine")

## k=5: SR only

In [ ]:
cfg_k5_sr = toric_sr_cfg(5, schedule="constant", seed=5, steps=3500)
params_k5_sr, hist_k5_sr, path_k5_sr = run_and_save(cfg_k5_sr, "toric4x4_k5_vit_sr")

## k=5: SR + cosine LR routine

In [ ]:
cfg_k5_sr_sched = toric_sr_cfg(5, schedule="cosine", seed=5, steps=3500)
params_k5_sr_sched, hist_k5_sr_sched, path_k5_sr_sched = run_and_save(cfg_k5_sr_sched, "toric4x4_k5_vit_sr_cosine")

## Compare energy and variance traces

In [ ]:
def collect(history):
    steps = np.array([r["step"] for r in history])
    energies = [r.get("energies", []) for r in history]
    maxk = max(len(x) for x in energies)
    E = np.full((len(history), maxk), np.nan)
    for i, row in enumerate(energies):
        E[i, :len(row)] = row

    variances = [r.get("state_energy_variances", []) for r in history]
    V = np.full((len(history), maxk), np.nan)
    for i, row in enumerate(variances):
        V[i, :len(row)] = row
    return steps, E, V

runs = {
    "k2 SR": hist_k2_sr,
    "k2 SR+cosine": hist_k2_sr_sched,
    "k4 SR": hist_k4_sr,
    "k4 SR+cosine": hist_k4_sr_sched,
    "k5 SR": hist_k5_sr,
    "k5 SR+cosine": hist_k5_sr_sched,
}

for name, hist in runs.items():
    steps, E, V = collect(hist)
    plt.figure()
    for i in range(E.shape[1]):
        plt.plot(steps, E[:, i], label=f"E{i}")
    plt.xlabel("step")
    plt.ylabel("Ritz-span energy")
    plt.title(name)
    plt.legend()
    plt.show()

    plt.figure()
    for i in range(V.shape[1]):
        plt.semilogy(steps, V[:, i], label=f"var{i}")
    plt.xlabel("step")
    plt.ylabel("NES S12 state energy variance")
    plt.title(name + " variance")
    plt.legend()
    plt.show()